# Baseline and Random Forest Models

This notebook establishes the first predictive benchmarks for the Cook County housing-price task. The primary evaluation set contains **2019 pure-market sales for properties not observed during 2013–2018**, so performance reflects forward generalization to unseen properties rather than memorization of repeated parcels.

The modeling sequence is deliberately incremental:

1. naive constant-price benchmark;
2. structural Random Forest using property characteristics only;
3. price-level calibration diagnostics;
4. location-aware Random Forest benchmark.

This first version implements the baseline and structural model. Later commits extend the same notebook with calibration and geography.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import build_analysis_dataset
from src.features import (
    PRIMARY_FEATURES,
    define_modeling_population,
    create_temporal_splits,
    make_model_matrices,
)
from src.modeling import (
    fit_dummy_baseline,
    predict_dummy_dollars,
    build_random_forest_pipeline,
    fit_log_target_model,
    predict_dollars,
)
from src.evaluation import regression_metrics, metrics_table

DATA_ZIP = PROJECT_ROOT / 'data' / 'raw' / 'cook_county_data.zip'

## 1. Reconstruct the Modeling Sample

The analysis dataset is rebuilt from the raw Cook County archive and ACS enrichment utilities. The same pure-market filter and temporal split defined in Notebook 2 are reused here so model comparison is performed on a fixed evaluation population.

In [ ]:
data_fair = build_analysis_dataset(DATA_ZIP)
model_data = define_modeling_population(data_fair)
train_data, test_data, repeated_property_test = create_temporal_splits(model_data)
matrices = make_model_matrices(train_data, test_data, repeated_property_test)

X_train = matrices['X_train']
X_test = matrices['X_test']
y_train = matrices['y_train']
y_test = matrices['y_test']
y_train_log = matrices['y_train_log']

print(f'Training sales: {len(X_train):,}')
print(f'Primary unseen-property test sales: {len(X_test):,}')
print(f'Structural predictors: {len(PRIMARY_FEATURES)}')

## 2. Naive Regression Baseline

Before fitting a flexible model, we establish a constant benchmark. The dummy model predicts the median training price on the log scale for every 2019 observation.

A useful predictive model should materially improve on this benchmark across absolute error, squared error, percentage error, and explained variance—not merely produce plausible-looking point predictions.

In [ ]:
dummy_model = fit_dummy_baseline(y_train_log)
dummy_pred = predict_dummy_dollars(dummy_model, len(X_test))

dummy_metrics = regression_metrics(y_test, dummy_pred)
pd.Series(dummy_metrics, name='Naive baseline')

## 3. Structural Random Forest

The first substantive model uses structural and transaction-timing variables while deliberately excluding fine-grained geography and ACS demographic attributes.

The preprocessing pipeline is fit only on historical training data. Numeric variables receive median imputation; categorical variables receive most-frequent imputation and one-hot encoding. The Random Forest is trained on `log(Sale Price)`, then predictions are transformed back to dollars for evaluation.

In [ ]:
structural_rf = build_random_forest_pipeline(include_location=False)
structural_rf = fit_log_target_model(structural_rf, X_train, y_train_log)
structural_pred = predict_dollars(structural_rf, X_test)

structural_metrics = regression_metrics(y_test, structural_pred)
pd.Series(structural_metrics, name='Structural Random Forest')

## 4. Baseline Comparison

The comparison below evaluates both models on exactly the same unseen-property 2019 holdout. The purpose of the structural Random Forest is not only to reduce average dollar error, but to establish how much predictive signal is available from property characteristics before location is introduced.

In [ ]:
comparison = metrics_table({
    'Naive baseline': (y_test, dummy_pred),
    'Structural Random Forest': (y_test, structural_pred),
})

comparison.style.format({
    'MAE': '${:,.0f}',
    'RMSE': '${:,.0f}',
    'MAPE': '{:.1f}%',
    'R2': '{:.3f}',
})

### Interpretation

In the completed full-project run, the naive unseen-property benchmark produced approximately **$181.7K MAE, $338.6K RMSE, and 91.9% MAPE**. The structural Random Forest reduced MAE to approximately **$109.2K** and achieved **R² ≈ 0.675**.

These results establish that structural housing characteristics contain substantial predictive signal, while still leaving enough unexplained variation to motivate a controlled geography benchmark in the next stage. Re-run this notebook in the project Codespace to regenerate the exact displayed metrics from the current pipeline.